# THE OBSERVER (Film II) — v4 Blueprint-First Pipeline

Generates its own character (no photos). Blueprint path: **perfect stills → animate with low denoise** → identity check → voice → score → sfx → stitch.

In [ ]:
# === SETUP ===
import os, sys, yaml, json, subprocess, shutil, urllib.request, zipfile, glob
os.chdir('/kaggle/working')

# Copy kit files from Dataset to working
kit_srcs = glob.glob('/kaggle/input/*')
kit_src = next((s for s in kit_srcs if os.path.isdir(os.path.join(s, 'src'))), None)
if kit_src:
    for item in os.listdir(kit_src):
        if item.startswith('.'): continue
        s = os.path.join(kit_src, item)
        d = os.path.join('/kaggle/working', item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
print('Kit root:', os.getcwd())


In [ ]:
# === DOWNLOAD WAN2.2-S2V Q8 GGUF ===
cfg = yaml.safe_load(open('config.yaml'))
models_dir = cfg['paths']['models']
os.makedirs(models_dir, exist_ok=True)

model_url = 'https://huggingface.co/Kijai/Wan2.1-GGUF/resolve/main/wan2.1-s2v-Q8_0.gguf'
model_path = os.path.join(models_dir, 'wan2.1-s2v-Q8_0.gguf')

if not os.path.exists(model_path):
    print('Downloading Wan2.2-S2V Q8 GGUF (~14GB)...')
    urllib.request.urlretrieve(model_url, model_path)
    print(f'Done: {os.path.getsize(model_path)/1e9:.1f} GB')
else:
    print(f'Cached: {os.path.getsize(model_path)/1e9:.1f} GB')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['KIT_CONFIG'] = 'config.yaml'
cfg['active_profile'] = 'kaggle_2xt4'
yaml.safe_dump(cfg, open('config.yaml', 'w'), sort_keys=False)
print('Profile: kaggle_2xt4 (FREE, 2x T4)')


In [ ]:
# STEP 1: preflight → FORGE Observer → render ONE sample shot → COST METER, then HOLD.
os.environ.setdefault('BUDGET_USD','4')
!python3 src/filmforge_observer.py

In [ ]:
# STEP 2: happy with the numbers? Uncomment to render the whole film:
# os.environ['CONFIRM_FULL_RUN']='1'
# !python3 src/filmforge_observer.py

In [ ]:
# STEP 3: package for download
# !cd output_observer && zip -qr /kaggle/working/THE_OBSERVER_final.zip THE_OBSERVER.mp4 THE_OBSERVER_MASTER_4444.mov 2>/dev/null; echo 'Download from Output tab.'

In [ ]:
# === QUALITY GATE + PACKAGE ===
!python3 src/qc.py

out_dir, targets = 'output', ['THE_MOMENT.mp4', 'THE_MOMENT_MASTER_4444.mov']
existing = [f for f in targets if os.path.exists(os.path.join(out_dir, f))]
zip_path = '/kaggle/working/THE_MOMENT_final.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in existing:
        z.write(os.path.join(out_dir, f), f)
print(f'Delivery zip: {zip_path}')
print('Download from Kaggle Output tab.')
